# 01. Prepare parent dataset

Collect and filter ChEMBL compounds, compute canonical SMILES, Murcko scaffolds,
and benchmark descriptors.

**Parent selection criteria:**
- Passes ChEMBL physicochemical filters (MW, HBD, HBA, RotB, ALogP)
- Valid RDKit parse
- Has a non-empty Murcko scaffold (required for scaffold-preserving benchmark)

The final CSV (`chembl_1000_parents.csv`) is used by all subsequent notebooks.


In [ ]:
import os, sys
import pandas as pd
from tqdm import tqdm
from rdkit import Chem
from rdkit.Chem import Descriptors

sys.path.append("..")

from configs.benchmark_config import (
    N_PARENTS, RANDOM_SEED, PARENT_DATASET,
    INDEPENDENT_TEST_DATASET,
    MW_MIN, MW_MAX, HBD_MAX, HBA_MAX, ROTB_MAX, ALOGP_MAX,
    N_PARENT_CANDIDATES_MULTIPLIER, SAVE_EVERY,
)
from utils.descriptors import calc_descriptors
from utils.scaffold import get_murcko_scaffold_smiles

print(f"Pandas: {pd.__version__}")


## 1. Collect from ChEMBL

In [ ]:
from chembl_webresource_client.settings import Settings
Settings.Instance().TIMEOUT = 60
Settings.Instance().TOTAL_RETRIES = 10
from chembl_webresource_client.new_client import new_client

molecule = new_client.molecule
results = molecule.filter(
    molecule_properties__mw_freebase__gte=MW_MIN,
    molecule_properties__mw_freebase__lte=MW_MAX,
    molecule_properties__hbd__lte=HBD_MAX,
    molecule_properties__hba__lte=HBA_MAX,
    molecule_properties__rtb__lte=ROTB_MAX,
    molecule_properties__alogp__lte=ALOGP_MAX,
    molecule_type="Small molecule",
).only([
    "molecule_chembl_id",
    "molecule_structures",
    "molecule_properties",
])


## 2. Parse, deduplicate, and save intermediate file

In [ ]:
INTERMEDIATE_FILE = "../data/chembl_parent_candidates.csv"

if os.path.exists(INTERMEDIATE_FILE):
    df_saved = pd.read_csv(INTERMEDIATE_FILE)
    records = df_saved.to_dict("records")
    seen_smiles = set(df_saved["smiles"].dropna())
    print(f"Resuming from {len(records)} saved compounds.")
else:
    records = []
    seen_smiles = set()
    print("Starting fresh.")

# Collect enough candidates to ensure N_PARENTS with valid scaffold after filtering.
target = N_PARENTS * N_PARENT_CANDIDATES_MULTIPLIER

try:
    for mol in tqdm(results, desc="Collecting"):
        if len(records) >= target:
            break
        try:
            smi = mol.get("molecule_structures", {}).get("canonical_smiles")
            cid = mol.get("molecule_chembl_id")
            if not smi or not cid:
                continue
            rdmol = Chem.MolFromSmiles(smi)
            if rdmol is None:
                continue
            canon = Chem.MolToSmiles(rdmol, canonical=True, isomericSmiles=True)
            if canon in seen_smiles:
                continue
            seen_smiles.add(canon)
            desc = calc_descriptors(rdmol)
            records.append({
                "chembl_id": cid,
                "smiles":    canon,
                "mw":        Descriptors.MolWt(rdmol),
                "logp":      Descriptors.MolLogP(rdmol),
                **desc,
            })
            if len(records) % SAVE_EVERY == 0:
                pd.DataFrame(records).to_csv(INTERMEDIATE_FILE, index=False)
                print(f"Saved: {len(records)}")
        except Exception:
            continue
except Exception as e:
    print(f"Interrupted: {e}")
finally:
    pd.DataFrame(records).to_csv(INTERMEDIATE_FILE, index=False)
    print(f"Total collected: {len(records)}")


## 3. Filter: require non-empty Murcko scaffold

Compounds without a meaningful Murcko scaffold are excluded from the benchmark
because scaffold-preservation cannot be evaluated for them.
After filtering, sample exactly N_PARENTS compounds.


In [ ]:
df_all = pd.read_csv(INTERMEDIATE_FILE)
df_all = df_all.drop_duplicates(subset="smiles").reset_index(drop=True)
print(f"Before scaffold filter: {len(df_all)}")

# Add Murcko scaffold SMILES; None means no valid scaffold.
def _scaffold_smi(smi):
    mol = Chem.MolFromSmiles(smi)
    return get_murcko_scaffold_smiles(mol) if mol else None

df_all["murcko_scaffold_smiles"] = df_all["smiles"].apply(_scaffold_smi)

# Keep only compounds with a non-empty scaffold.
df_valid = df_all.dropna(subset=["murcko_scaffold_smiles"]).reset_index(drop=True)
print(f"After scaffold filter:  {len(df_valid)}")

if len(df_valid) < N_PARENTS:
    raise RuntimeError(
        f"Only {len(df_valid)} compounds with valid scaffold available; "
        f"need {N_PARENTS}. Increase N_PARENT_CANDIDATES_MULTIPLIER or relax filters."
    )

df = df_valid.sample(n=N_PARENTS, random_state=RANDOM_SEED).reset_index(drop=True)
print(f"Sampled: {len(df)}")


## 4. Build final parent CSV

In [ ]:
# Unified column naming used throughout the benchmark.
df = df.rename(columns={
    "chembl_id": "parent_id",
    "smiles":    "canonical_smiles",
})

# Recalculate descriptors from canonical SMILES to be safe.
def _desc(smi):
    mol = Chem.MolFromSmiles(smi)
    return calc_descriptors(mol) if mol else {"hbd": None, "hba": None, "ar": None, "rotb": None}

descs = df["canonical_smiles"].apply(_desc).apply(pd.Series)
for col in descs.columns:
    df[col] = descs[col]

# Column order.
cols_front = ["parent_id", "canonical_smiles", "murcko_scaffold_smiles",
              "hbd", "hba", "ar", "rotb", "mw", "logp"]
df = df[[c for c in cols_front if c in df.columns] +
        [c for c in df.columns if c not in cols_front]]

OUTPUT_FILE = PARENT_DATASET
df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(df)} parents → {OUTPUT_FILE}")
df.head()


## 5. Sanity check

In [ ]:
df_check = pd.read_csv(OUTPUT_FILE)
print(df_check.shape)
assert df_check["murcko_scaffold_smiles"].isna().sum() == 0, "Null scaffolds remain!"
assert len(df_check) == N_PARENTS, f"Expected {N_PARENTS} rows, got {len(df_check)}"
print("All checks passed.")
df_check[["parent_id", "canonical_smiles", "murcko_scaffold_smiles",
          "hbd", "hba", "ar", "rotb"]].head()


## 6. Verify no overlap with independent test set (if it exists)

In [ ]:
import os
if os.path.exists(INDEPENDENT_TEST_DATASET):
    test_df = pd.read_csv(INDEPENDENT_TEST_DATASET)
    test_col = "canonical_smiles" if "canonical_smiles" in test_df.columns else "smiles"
    dev_col  = "canonical_smiles" if "canonical_smiles" in df_check.columns else "smiles"
    overlap  = set(df_check[dev_col]) & set(test_df[test_col])
    assert len(overlap) == 0, f"Overlap detected: {len(overlap)} shared SMILES!"
    print(f"Development set : {len(df_check)} parents")
    print(f"Independent test: {len(test_df)} parents")
    print(f"Overlap         : {len(overlap)} (required: 0)")
    print("✓ No overlap.")
else:
    print(f"Independent test set not yet created: {INDEPENDENT_TEST_DATASET}")
    print("Run 00_prepare_independent_test.ipynb first.")
